# 10. 공개 언어모델 생성 문장으로 어휘 증강
09 (검증 0.9908) 에 남은 글자 오류 1,026개:

| 원인 | 비중 | 정답 어절이 train 에 없는 비율 |
|---|---|---|
| ① 고정했는데 틀림 | 31% | 41% |
| ② 후보에 정답이 없음 | 36% | **89%** |
| ③ 언어모델이 잘못 고름 | 33% | 75% |

반납하고·기념일·혼잡하다·찝찝하고·귀중품처럼 흔한 말인데 train 리뷰 1만 개에 없던 단어를 01(노이즈 채널)이 모른다.
train 리뷰로 미세조정한 Qwen2.5-1.5B 가 생성한 리뷰를 **01 의 사전과 문자 n-gram 에 추가**해 어휘를 넓힌다.

1. 미세조정 모델로 리뷰 생성 (`lm_generate.py`) → `outputs/gen_reviews_split.txt`
2. 01 만 다시 검증 (CPU, 수 분): 생성 문장을 LM 에 몇 개 넣을지 × 사전 가중치 탐색. **(0, 0) = 기존 01 (0.9834)**
3. 01 이 좋아졌으면 그 위에서 09 파이프라인(고정 + 문맥 추론) 재실행

10단계 결과는 `outputs10/` 에 따로 저장 (기존 `outputs/` 의 01 예측·후보 캐시와 섞이지 않게).
검증용은 train 90% 로 미세조정한 모델과 그 90% 문장만 사용 → 검증 정답 누출 없음.

**GPU 런타임(L4 또는 A100), 첫 셀부터.**

## 0) 준비

In [ ]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' --exclude 'v2_cache_*.pkl' "$DRIVE/" /content/work/
%cd /content/work
!mkdir -p outputs10 && ls -d outputs/lm_adapter_split outputs/lm_adapter_full 2>/dev/null; nproc; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    !rsync -a --update --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    !mkdir -p "$DRIVE/outputs10" && rsync -a --update /content/work/outputs10/ "$DRIVE/outputs10/"
    print('saved outputs, outputs10 -> drive')

## 1) 미세조정 모델 준비 (학습 없이 저장된 어댑터로 조립, 1~2분)

In [ ]:
!pip -q install peft
!pip -q uninstall -y torchao
!python -u lm_finetune.py --merge_only

## 2) 리뷰 생성 (검증용)
출력 마지막 줄 '검증 정답 어절 중 train 에 없는 것 → 생성 문장이 덮는 것' 이 증강 효과의 선행 지표.
새 어절 예시와 생성 문장 예시가 리뷰답게 나오는지도 눈으로 확인.

In [ ]:
!python -u lm_generate.py --n 100000
save_back()


## 3) 01 에 증강 적용: LM 에 넣을 문장 수 × 사전 가중치 탐색 (CPU)
- `aug_lm_n` : 문자 n-gram LM 에 추가할 생성 문장 수
- `aug_dict_weight` : 사전에 추가하는 새 어절(생성 문장에 2회 이상 등장, train 에 없던 것)의 빈도 가중치
- 첫 조합 (0, 0) 은 기존 01 → **0.9834 가 재현되면 정상**
- 최고 설정은 `outputs10/best_params.json`, 그 검증 예측은 `outputs10/val_pred.csv`

In [ ]:
!python -u noisy_channel.py --mode sweep --out_dir outputs10 --lm_weight 1.1 --beam 16 --dict_oov 0.5 \
    --aug_file outputs/gen_reviews_split.txt --grid "aug_lm_n=30000,60000,100000;aug_dict_weight=0.3,1"
save_back()

In [ ]:
import pandas as pd
r = pd.read_csv('outputs10/nc_runs.csv')
print(r.pivot_table(index='aug_lm_n', columns='aug_dict_weight', values='f1', aggfunc='max').round(5))
print(open('outputs10/best_params.json', encoding='utf-8').read())

## 4) (3 에서 01 이 좋아졌을 때만) 증강한 01 위에서 09 파이프라인 재실행
3 의 최고가 (0, 0) 이면 증강 효과가 없는 것이므로 여기서 멈춘다.
v2 태거 예측은 기존 것을 그대로 참조로 쓴다. 비교 기준: 09 = 0.9908.

In [ ]:
!cp outputs/val_pred_tagger_v2_scratch.csv outputs/submission_tagger_v2_scratch.csv outputs10/
!python -u anchor_infill.py --mode val --out_dir outputs10 --weights 0.9,1.0,1.1 --unanchor_oov --cand_beam 64 --nbest 20 --batch_tokens 8192 --lm models/qwen_ft_split --name 10
save_back()

In [ ]:
a = pd.read_csv('outputs/val_pred.csv').set_index('ID')
n = pd.read_csv('outputs10/val_pred.csv').set_index('ID')
b9 = pd.read_csv('outputs/val_pred_anchor_agree_v2_k20_b64_uo_09abc.csv').set_index('ID')
b10 = pd.read_csv('outputs10/val_pred_anchor_agree_v2_k20_b64_uo_10.csv').set_index('ID')
print(f'01            {a.f1.mean():.4f}\n01 + 증강     {n.f1.mean():.4f}\n09            {b9.f1.mean():.4f}\n10            {b10.f1.mean():.4f}')
d = b10.f1 - b9.f1
print(f'10 vs 09: 좋아진 행 {(d>0).sum()} / 나빠진 행 {(d<0).sum()}')
for i in d.sort_values().index[:3].tolist() + d.sort_values().index[-4:].tolist():
    print(f'[{d[i]:+.3f}] GT :', b10.loc[i, 'output'][:90]); print('        09 :', str(b9.loc[i, 'pred'])[:90]); print('        10 :', str(b10.loc[i, 'pred'])[:90])

## 5) test 예측 (4 가 09 보다 좋을 때)
test 용은 train 전체로 미세조정한 모델로 문장을 다시 생성한다 (어댑터 `lm_adapter_full` 은 09 에서 저장됨).
결과: `outputs10/submission_anchor_agree_v2_k20_b64_uo_10.csv`

In [ ]:
!pip -q install peft
!pip -q uninstall -y torchao
!python -u lm_finetune.py --full --merge_only
!python -u lm_generate.py --full --n 100000
!python -u noisy_channel.py --mode test --out_dir outputs10 --aug_file outputs/gen_reviews_full.txt
!python -u anchor_infill.py --mode test --out_dir outputs10 --aug_file outputs/gen_reviews_full.txt --unanchor_oov --cand_beam 64 --nbest 20 --batch_tokens 32768 --lm models/qwen_ft_full --name 10
save_back()

### (선택) 런타임 해제

In [ ]:
from google.colab import runtime; runtime.unassign()